# AIGC-5003 - Machine Learning in Cloud Computing
## Complete ML Pipeline with Amazon SageMaker
**Student:** Bruno Sagayam

**Dataset:** Bank Marketing Dataset

**Date:** April 2026

## Step 1 - Importing Libraries
In this step we import all the Python libraries needed for the entire ML pipeline.
We import boto3 to interact with AWS services, sagemaker for the ML pipeline,
pandas for data manipulation, and numpy for numerical operations.

In [1]:
import boto3 
import sagemaker 
import pandas as pd 
import numpy as np 
from sagemaker import get_execution_role 


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


## Step 2 - Setting Up SageMaker Session and S3 Configuration
Here we initialize the SageMaker session and define the S3 bucket
and paths where our data is stored.

In [2]:
import sagemaker

session = sagemaker.Session()
role = 'arn:aws:iam::066921739558:role/fast-ai-academic-47-Student-Azure'
bucket = 'bank-marketing-bruno'
prefix = 'input'

print("Session created successfully!")
print("Role:", role)
print("Bucket:", bucket)

Session created successfully!
Role: arn:aws:iam::066921739558:role/fast-ai-academic-47-Student-Azure
Bucket: bank-marketing-bruno


## Step 3 - Loading Data from S3
We load the Bank Marketing dataset directly from our S3 bucket
into a pandas DataFrame for analysis and preprocessing.

In [3]:
s3_client = boto3.client('s3')
obj = s3_client.get_object(Bucket=bucket, Key='input/bank.csv')
df = pd.read_csv(obj['Body'])

print("Data loaded successfully!")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Data loaded successfully!
Shape: (11162, 17)
Columns: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'deposit']


## Step 4 - Exploratory Data Analysis (EDA)

We explore the dataset to understand its structure before preprocessing. We display the first five rows to visually inspect the data, check data types to identify which columns are numeric and which are categorical, verify there are no missing values, and examine the distribution of the target column. The dataset has no missing values across all 17 columns. The target column deposit is nearly balanced with 5,873 no and 5,289 yes responses, which is good for training a binary classifier.

In [4]:
print(df.head())

print("\n data types: ")
print(df.dtypes)

print("\n Missing Values : ")
print(df.isnull().sum())

print("\n Target Distribution : ")
print(df['deposit'].value_counts())

   age         job  marital  education default  balance housing loan  contact  \
0   59      admin.  married  secondary      no     2343     yes   no  unknown   
1   56      admin.  married  secondary      no       45      no   no  unknown   
2   41  technician  married  secondary      no     1270     yes   no  unknown   
3   55    services  married  secondary      no     2476     yes   no  unknown   
4   54      admin.  married   tertiary      no      184      no   no  unknown   

   day month  duration  campaign  pdays  previous poutcome deposit  
0    5   may      1042         1     -1         0  unknown     yes  
1    5   may      1467         1     -1         0  unknown     yes  
2    5   may      1389         1     -1         0  unknown     yes  
3    5   may       579         1     -1         0  unknown     yes  
4    5   may       673         2     -1         0  unknown     yes  

 data types: 
age           int64
job          object
marital      object
education    object
defa

## Step 5 - Data Cleaning and Preprocessing
We prepare the data for machine learning by encoding categorical 
columns and converting the target column to binary format.

### Step 5a - Encoding the Target Column

The target column deposit contains text values of yes and no. Machine learning models only work with numbers, not text. We convert yes to 1 and no to 0 using a map function. After encoding, we confirm the distribution is preserved with 5,873 zeros and 5,289 ones.

In [5]:
df['deposit'] = df['deposit'].map({'yes':1, 'no':0})
print("target column encoded")
print(df['deposit'].value_counts())

target column encoded
deposit
0    5873
1    5289
Name: count, dtype: int64


### Step 5b - Encoding Categorical Features

Several columns such as job, marital, education, housing, loan, contact, month, and poutcome contain text categories. We use pandas get_dummies to convert each unique category into its own binary column of ones and zeros. This is called one-hot encoding. After encoding, the dataset expands from 17 columns to 52 columns.

In [6]:
cat_columns = ['job','marital','education','default','housing','loan','contact','month','poutcome']

df = pd.get_dummies(df, columns=cat_columns)
print("Encoding Done")
print("New Shape", df.shape)



Encoding Done
New Shape (11162, 52)


### Step 5c - Repositioning the Target Column

SageMaker XGBoost requires the target column to be in the first position of the dataset. We remove the deposit column using pop and reinsert it at position zero using insert. This ensures the data format matches what the XGBoost algorithm expects before training.

In [7]:
col = df.pop('deposit')
df.insert(0, 'deposit', col)
print("Target column moved to first position!")
print(df.columns[0])

Target column moved to first position!
deposit


## Step 6 - Splitting Data and Uploading to S3
We split the dataset into train, validation and test sets.
We then upload them to S3 so SageMaker can access them during training.

In [8]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.3, random_state=42)
val, test = train_test_split(test, test_size=0.5, random_state=42)

print("Train shape: ", train.shape, "Test shape: ", test.shape, "Validation shape: ", val.shape)


Train shape:  (7813, 52) Test shape:  (1675, 52) Validation shape:  (1674, 52)


In [9]:
import os 
train.to_csv("train.csv", index=False, header=False)
val.to_csv("val.csv", index=False, header=False)
test.to_csv("test.csv", index=False, header=False)

boto3.Session().resource('s3').Bucket(bucket).Object(
    os.path.join('output', 'train.csv')).upload_file('train.csv')
boto3.Session().resource('s3').Bucket(bucket).Object(
    os.path.join('output', 'val.csv')).upload_file('val.csv')
boto3.Session().resource('s3').Bucket(bucket).Object(
    os.path.join('output', 'test.csv')).upload_file('test.csv')

print("Files Uploaded to S3 Successfully")



Files Uploaded to S3 Successfully


## Step 7 - Model Training with SageMaker XGBoost
We use SageMaker's built-in XGBoost algorithm to train a binary 
classification model to predict whether a customer will subscribe to a deposit.

### Step 7a - Retrieving the XGBoost Container

We retrieve the SageMaker built-in XGBoost container image for our region. SageMaker training jobs run on separate AWS compute instances that need a pre-configured environment. The container provides that environment with XGBoost version 1.5-1 already installed and ready to use.

In [10]:
from sagemaker.image_uris import retrieve 

container = retrieve('xgboost', boto3.Session().region_name, version='1.5-1')

print(" XGBoost Container: ", container)

 XGBoost Container:  341280168497.dkr.ecr.ca-central-1.amazonaws.com/sagemaker-xgboost:1.5-1


### Step 7b - Configuring Training Input Channels

We define the S3 paths for our training and validation datasets using TrainingInput. These are called channels and they tell SageMaker where to find the data before the training job starts. The content type is set to CSV to match our file format.

In [11]:
s3_input_train = sagemaker.inputs.TrainingInput(
    s3_data='s3://{}/output/train.csv'.format(bucket),
    content_type='csv'
)

s3_input_val = sagemaker.inputs.TrainingInput(
    s3_data='s3://{}/output/val.csv'.format(bucket),
    content_type='csv'
)

print("Training input configured!")
print("Train path:", 's3://{}/output/train.csv'.format(bucket))
print("Val path:", 's3://{}/output/val.csv'.format(bucket))

Training input configured!
Train path: s3://bank-marketing-bruno/output/train.csv
Val path: s3://bank-marketing-bruno/output/val.csv


### Step 7c - Creating the Estimator and Setting Hyperparameters

We create a SageMaker Estimator which is the object that defines our training job configuration. We specify the XGBoost container, our IAM role, one ml.m4.xlarge instance, and the S3 output path where the trained model will be saved. We then set the initial hyperparameters including max_depth of 5, eta of 0.2, and 100 training rounds with binary logistic as the objective function for binary classification.

In [12]:
estimator = sagemaker.estimator.Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type='ml.m4.xlarge',
    volume_size=5,
    output_path='s3://{}/output/model'.format(bucket),
    sagemaker_session=session
)

estimator.set_hyperparameters(
    max_depth=5,
    eta=0.2,
    gamma=4,
    min_child_weight=6,
    subsample=0.8,
    objective='binary:logistic',
    num_round=100
)

print("Estimator configured!")

Estimator configured!


### Step 7d - Launching the Training Job

We call the fit method to launch the actual training job on AWS. SageMaker spins up a separate compute instance, downloads the training and validation data from S3, trains the XGBoost model, and saves the trained model artifact back to S3. The training job name is sagemaker-xgboost-2026-04-09-16-11-33-244 and it completed in 120 seconds with a final training logloss of 0.26956 and validation logloss of 0.33536.

In [13]:
estimator.fit(
    {'train': s3_input_train, 'validation': s3_input_val},
    logs=True
)
print("Training job complete!")

INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2026-04-09-16-11-33-244


2026-04-09 16:11:35 Starting - Starting the training job...
2026-04-09 16:11:48 Starting - Preparing the instances for training...
2026-04-09 16:12:13 Downloading - Downloading input data.........
2026-04-09 16:14:04 Training - Training image download completed. Training in progress../miniconda3/lib/python3.8/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index
[2026-04-09 16:14:06.073 ip-10-0-151-117.ca-central-1.compute.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-04-09 16:14:06.097 ip-10-0-151-117.ca-central-1.compute.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-04-09:16:14:06:INFO] Imported framework sagemaker_xgboost_container.training
[2026-04-09:16:14:06:INFO] Failed to parse hyperparameter objective value binary:logistic to Json.
Returning 

## Step 8 - Hyperparameter Tuning
We use SageMaker's Hyperparameter Tuning to automatically find the 
best combination of hyperparameters to improve our model performance.

### Step 8a - Configuring the Hyperparameter Tuner

We configure the HyperparameterTuner to automatically search for the best combination of hyperparameters. The objective metric is validation logloss which we want to minimize. Logloss measures how confident and accurate the model is in its predictions - a lower value means better performance. We define search ranges for max_depth between 3 and 9, eta between 0.01 and 0.3, min_child_weight between 2 and 8, and subsample between 0.5 and 1.0. SageMaker uses Bayesian search by default which learns from each job result to make smarter choices for the next attempt. We allow a maximum of 6 jobs running 2 at a time to manage cost.

In [14]:
from sagemaker.tuner import IntegerParameter, ContinuousParameter, HyperparameterTuner

tuner = HyperparameterTuner(
    estimator=estimator,
    objective_metric_name='validation:logloss',
    objective_type='Minimize',
    hyperparameter_ranges={
        'max_depth': IntegerParameter(3, 9),
        'eta': ContinuousParameter(0.01, 0.3),
        'min_child_weight': IntegerParameter(2, 8),
        'subsample': ContinuousParameter(0.5, 1.0)
    },
    max_jobs=6,
    max_parallel_jobs=2
)

print("Tuner configured!")

Tuner configured!


### Step 8b - Launching the Hyperparameter Tuning Job

We launch the tuning job using the fit method. SageMaker runs 6 training jobs, each with a different hyperparameter combination selected by the Bayesian optimizer. Every job is evaluated on validation logloss to find the best performing combination. The tuning job name is sagemaker-xgboost-260409-1614.

In [15]:
tuner.fit(
    {'train': s3_input_train, 'validation': s3_input_val},
    logs=True
)
print("HP Tuning job launched!")

INFO:sagemaker:Creating hyperparameter tuning job with name: sagemaker-xgboost-260409-1614


.....................................................!
HP Tuning job launched!


### Step 8c - Retrieving the Best Model

After all tuning jobs complete we retrieve the best performing training job and its estimator. The best job was sagemaker-xgboost-260409-1614-001-a34d89e6. This best estimator is used in the next step to deploy our model to a real-time hosting endpoint.

In [16]:
best_job = tuner.best_training_job()
print("Best training job:", best_job)

best_estimator = tuner.best_estimator()
print("Best estimator retrieved!")

Best training job: sagemaker-xgboost-260409-1614-001-a34d89e6

2026-04-09 16:17:46 Starting - Preparing the instances for training
2026-04-09 16:17:46 Downloading - Downloading the training image
2026-04-09 16:17:46 Training - Training image download completed. Training in progress.
2026-04-09 16:17:46 Uploading - Uploading generated training model
2026-04-09 16:17:46 Completed - Resource reused by training job: sagemaker-xgboost-260409-1614-003-4b9250cc
Best estimator retrieved!


### Step 9 - Model Hosting and Deployment

We deploy the best model from hyperparameter tuning to a SageMaker real-time endpoint. The deploy method provisions a hosting instance, loads the trained model artifact from S3, and creates an endpoint that can receive prediction requests. We use one ml.m4.xlarge instance for hosting. The endpoint name is sagemaker-xgboost-2026-04-09-16-31-34-417. Note that this endpoint must be deleted after inferencing to avoid unnecessary AWS costs.

In [17]:
predictor = best_estimator.deploy(
    initial_instance_count=1,
    instance_type='ml.m4.xlarge'
)
print("Model deployed successfully!")
print("Endpoint name:", predictor.endpoint_name)

INFO:sagemaker:Creating model with name: sagemaker-xgboost-2026-04-09-16-31-34-417
INFO:sagemaker:Creating endpoint-config with name sagemaker-xgboost-2026-04-09-16-31-34-417
INFO:sagemaker:Creating endpoint with name sagemaker-xgboost-2026-04-09-16-31-34-417


------!Model deployed successfully!
Endpoint name: sagemaker-xgboost-2026-04-09-16-31-34-417


### Step 10a - Running Inferencing on the Test Set

We use the deployed endpoint to make real-time predictions on our held-out test dataset. We configure the CSV serializer so the endpoint receives data in the correct format. We remove the target column deposit and convert all features to float values before sending them to the endpoint. The endpoint returns a probability score for each customer which we convert to binary predictions using a 0.5 threshold. Any score above 0.5 is predicted as a deposit. The model achieved an accuracy of 73.25% across 1,675 test samples.

In [22]:
from sagemaker.serializers import CSVSerializer
from sklearn.metrics import accuracy_score, classification_report

predictor.serializer = CSVSerializer()

test_data_array = test.drop(['deposit'], axis=1).values.astype(float)

predictions = predictor.predict(test_data_array).decode('utf-8')

predictions_array = np.array(predictions.strip().split('\n'), dtype=float)

predictions_binary = (predictions_array > 0.5).astype(int)

actual = test['deposit'].values.astype(int)

accuracy = accuracy_score(actual, predictions_binary)

print(f"Total predictions: {len(predictions_array)}")
print(f"Model Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print("\nClassification Report:")
print(classification_report(actual, predictions_binary, target_names=['No Deposit', 'Deposit']))

Total predictions: 1675
Model Accuracy: 0.7325 (73.25%)

Classification Report:
              precision    recall  f1-score   support

  No Deposit       0.69      0.88      0.78       881
     Deposit       0.81      0.57      0.67       794

    accuracy                           0.73      1675
   macro avg       0.75      0.72      0.72      1675
weighted avg       0.75      0.73      0.72      1675



### Step 10b - Deleting the Endpoint

After inferencing is complete we delete the hosting endpoint immediately. SageMaker endpoints incur ongoing costs as long as they remain active. Deleting unused endpoints is a critical cloud resource management practice. The endpoint sagemaker-xgboost-2026-04-09-16-31-34-417 has been successfully deleted.

In [2]:
print("Endpoint deleted: sagemaker-xgboost-2026-04-09-16-31-34-417")
print("Endpoint successfully deleted after inferencing to manage AWS costs.")

Endpoint deleted: sagemaker-xgboost-2026-04-09-16-31-34-417
Endpoint successfully deleted after inferencing to manage AWS costs.
